# 83 — Transductive Graph Label Spreading

Build a molecular similarity graph over ALL compounds (train + test).
Propagate known pEC50 labels from training nodes to unlabeled test nodes
through similarity edges — like PageRank but for activity prediction.

Why this works: test = analog expansion of 63 hits → dense edges between
test and training nodes → labels diffuse efficiently through the graph.

Uses sklearn LabelSpreading with a Tanimoto-affinity kernel.
The spread predictions are then blended with a direct LGBM model.


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and len(cp)>0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
from sklearn.semi_supervised import LabelSpreading

# Build affinity matrix over train+test (using top-k sparse Tanimoto)
n_tr = len(tr); n_te = len(te); N = n_tr + n_te
fps_all = np.vstack([fps_tr, fps_te]).astype(np.float32)
y_all_labeled = np.concatenate([y_tr, np.full(n_te, np.nan)])

print(f"Building Tanimoto affinity matrix for {N} compounds...", flush=True)
# Compute in batches to avoid OOM
BATCH = 200
K_TOP = 10   # keep top-K neighbors per compound for sparse graph
rows, cols, vals = [], [], []

for start in range(0, N, BATCH):
    end = min(start + BATCH, N)
    fps_batch = fps_all[start:end]
    dot = fps_batch @ fps_all.T
    rs_b = fps_batch.sum(1)[:,None]; rs_a = fps_all.sum(1)[None,:]
    sim_block = dot / np.maximum(rs_b + rs_a - dot, 1e-6)
    np.fill_diagonal(sim_block[:,start:end], 0)  # no self-loops
    for i, sim_row in enumerate(sim_block):
        top_k = np.argpartition(sim_row, -K_TOP)[-K_TOP:]
        top_k = top_k[sim_row[top_k] > 0.3]  # threshold
        for j in top_k:
            rows.append(start + i); cols.append(j); vals.append(float(sim_row[j]))
    if (start//BATCH) % 10 == 0:
        print(f"  {end}/{N}", flush=True)

from scipy.sparse import csr_matrix
A = csr_matrix((vals, (rows, cols)), shape=(N, N))
A = A.maximum(A.T)  # symmetrize
print(f"Graph: {N} nodes, {A.nnz} edges, density={A.nnz/(N*N)*100:.3f}%")


Building Tanimoto affinity matrix for 4652 compounds...


  200/4652


  2200/4652


  4200/4652


Graph: 4652 nodes, 34566 edges, density=0.160%


In [5]:
# LabelSpreading expects dense affinity — use our sparse matrix directly
# For large N, use manual iteration (kernel=precomputed)
print("Running label spreading...", flush=True)
ls = LabelSpreading(kernel="rbf", alpha=0.8, max_iter=100, tol=1e-4)

# Encode: labeled=0..max_class (regression hack: bin pEC50 into 20 classes)
N_BINS = 20
y_min, y_max = y_tr.min(), y_tr.max()
bin_edges = np.linspace(y_min, y_max, N_BINS+1)
y_tr_binned = np.digitize(y_tr, bin_edges[1:-1])  # 0 to N_BINS-1
y_ls_input = np.concatenate([y_tr_binned, np.full(n_te, -1)])  # -1 = unlabeled

# Use dense Tanimoto matrix (for n<5000 this is feasible)
if N <= 5000:
    A_dense = A.toarray().astype(np.float64)
    ls.fit(A_dense, y_ls_input)
    y_spread_bins = ls.transduction_
else:
    # Manual power iteration for larger graphs
    print("  Too large for dense — using sparse power iteration")
    # Initialize: training nodes fixed, test nodes = training mean
    f = np.zeros(N); f[:n_tr] = y_tr; f[n_tr:] = y_tr.mean()
    D = np.array(A.sum(1)).flatten()
    D_inv = np.where(D>0, 1/D, 0)
    alpha_ls = 0.8
    for it in range(30):
        f_new = alpha_ls * (A.dot(f) * D_inv) + (1-alpha_ls) * np.where(
            y_ls_input >= 0, y_tr.mean() + (y_ls_input - y_tr_binned.mean()) * (y_max-y_min)/N_BINS, 0)
        f_new[:n_tr] = y_tr  # clamp labeled nodes
        if np.max(np.abs(f_new - f)) < 1e-4: break
        f = f_new
    y_spread = f
    y_spread_bins = None

print("Label spreading done.")


Running label spreading...


Label spreading done.


In [6]:
# Extract test predictions from label spreading
if y_spread_bins is not None:
    # Convert bins back to pEC50
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    y_spread_all = bin_centers[np.clip(y_spread_bins, 0, N_BINS-1)]
else:
    y_spread_all = y_spread

te_spread = y_spread_all[n_tr:]
print(f"Spread predictions range: [{te_spread.min():.2f}, {te_spread.max():.2f}]")

# Compare with direct LGBM as OOF on training nodes
oof_direct = np.full(n_tr, np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                  valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m.predict(X_tr[va_idx])

m_dir = full_metrics(y_tr, oof_direct, cliff_pairs, "direct_lgbm")

# Spread OOF: check spread quality on training (labeled) nodes
y_spread_tr = y_spread_all[:n_tr]
m_spread = full_metrics(y_tr, y_spread_tr, cliff_pairs, "spread_train_check")

# Blend spread + direct for test predictions
m_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

# Weight by spread confidence (how densely connected is each test node)
te_density = np.array(A[n_tr:].sum(1)).flatten()
te_density_norm = te_density / (te_density.max() + 1e-8)

alpha_s = 0.3  # weight for spread prediction
te_preds = (alpha_s * te_spread * te_density_norm + (1-alpha_s) * te_direct)
te_preds = np.clip(te_preds / (alpha_s * te_density_norm + (1-alpha_s)),
                   y_tr.min()-0.5, y_tr.max()+0.5)
oof = oof_direct  # OOF from LGBM (spread doesn't have proper CV OOF)
print(pd.DataFrame([m_dir, m_spread],index=["direct","spread"]).round(4).to_string())
np.save(DATA_PROCESSED/"oof_graph_spreading.npy", oof)
np.save(DATA_PROCESSED/"te_oof_graph_spreading.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"83_graph_label_spreading.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


Spread predictions range: [2.65, 6.81]


  [direct_lgbm] RAE=0.5643 MAE=0.5134 R²=0.5991 r=0.7740 ρ=0.7268 τ=0.5345  Cliff=nan
  [spread_train_check] RAE=0.0824 MAE=0.0750 R²=0.9935 r=0.9968 ρ=0.9938 τ=0.9525  Cliff=nan


           RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
direct  0.5643  0.5134  0.5991   0.7740    0.7268   0.5345        NaN
spread  0.0824  0.0750  0.9935   0.9968    0.9938   0.9525        NaN
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\83_graph_label_spreading.csv
Test: min=3.07 med=5.03 max=6.01
